# 01 — Data Loading and Schema Validation

This notebook loads the four raw analytical datasets and checks their structure before any
substantive cleaning is performed:

1. Census 2021 TS045 car/van availability;
2. Index of Deprivation 2025, File 7;
3. the September 2025 EVSE location registry; and
4. the September 2025 charging-activity file.

The substantive filtering, de-duplication, interval merging, zero-session treatment, spatial
join and LSOA aggregation are performed in `02_data_cleaning_revised.ipynb`.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/Users/alexia/Documents/CASA/Dissertation")

PATHS = {
    "ts045": BASE / "03_data/demand/census2021/TS045_car_van_availability_London_LSOA_2021.csv",
    "iod2025": BASE / "03_data/demand/IoD2025.csv",
    "evse_location": BASE / "03_data/restricted/evse_location.csv",
    "charging_activity": BASE / "03_data/restricted/charging_activity_sep.csv",
    "lsoa_boundaries": BASE / "03_data/demand/spatial/LSOA_2021_EW_BGC_V5.shp",
}

for name, path in PATHS.items():
    print(f"{name:<20} {path} | exists={path.exists()}")

missing_files = [str(path) for path in PATHS.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing required file(s):\n" + "\n".join(missing_files))


ts045                /Users/alexia/Documents/CASA/Dissertation/03_data/demand/census2021/TS045_car_van_availability_London_LSOA_2021.csv | exists=True
iod2025              /Users/alexia/Documents/CASA/Dissertation/03_data/demand/IoD2025.csv | exists=True
evse_location        /Users/alexia/Documents/CASA/Dissertation/03_data/restricted/evse_location.csv | exists=True
charging_activity    /Users/alexia/Documents/CASA/Dissertation/03_data/restricted/charging_activity_sep.csv | exists=True
lsoa_boundaries      /Users/alexia/Documents/CASA/Dissertation/03_data/demand/spatial/LSOA_2021_EW_BGC_V5.shp | exists=True


In [2]:
def require_columns(df: pd.DataFrame, required: list[str], dataset_name: str) -> None:
    """Raise a clear error when an expected source column is missing."""
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f"{dataset_name}: missing required columns: {missing}")


## 1. Census 2021 TS045


In [3]:
ts045_raw = pd.read_csv(PATHS["ts045"], skiprows=7)

require_columns(
    ts045_raw,
    [
        "Area",
        "No cars or vans in household",
        "1 car or van in household",
        "2 cars or vans in household",
        "3 or more cars or vans in household",
    ],
    "TS045",
)

ts045_raw = ts045_raw.rename(columns={
    "Area": "area_raw",
    "No cars or vans in household": "cars_0",
    "1 car or van in household": "cars_1",
    "2 cars or vans in household": "cars_2",
    "3 or more cars or vans in household": "cars_3plus",
})

# The Nomis export contains a disclosure-control footer. Rows without an Area value are not data.
ts045_raw = ts045_raw.dropna(subset=["area_raw"]).copy()
for col in ["cars_0", "cars_1", "cars_2", "cars_3plus"]:
    ts045_raw[col] = pd.to_numeric(ts045_raw[col], errors="coerce")

ts045_gor_london = ts045_raw.loc[ts045_raw["area_raw"].eq("gor:London")].copy()
ts045_lsoa = ts045_raw.loc[
    ts045_raw["area_raw"].str.startswith("lsoa2021:", na=False)
].copy()

parsed = ts045_lsoa["area_raw"].str.extract(
    r"lsoa2021:(?P<lsoa_code>\S+)\s*:\s*(?P<lsoa_name>.+)"
)
ts045_lsoa[["lsoa_code", "lsoa_name"]] = parsed[["lsoa_code", "lsoa_name"]]

ts045 = ts045_lsoa[
    ["lsoa_code", "lsoa_name", "cars_0", "cars_1", "cars_2", "cars_3plus"]
].reset_index(drop=True)

print("TS045 LSOA rows:", f"{len(ts045):,}")
print("Duplicate LSOA codes:", ts045["lsoa_code"].duplicated().sum())
print("Missing parsed LSOA codes:", ts045["lsoa_code"].isna().sum())
display(ts045.head())


TS045 LSOA rows: 35,672
Duplicate LSOA codes: 0
Missing parsed LSOA codes: 0


,lsoa_code,lsoa_name,cars_0,cars_1,cars_2,cars_3plus
0,E01011954,Hartlepool 001A,305,388.0,208.0,62.0
1,E01011969,Hartlepool 001B,87,296.0,168.0,53.0
2,E01011970,Hartlepool 001C,48,231.0,153.0,53.0
3,E01011971,Hartlepool 001D,29,201.0,204.0,86.0
4,E01033465,Hartlepool 001F,56,282.0,323.0,79.0


## 2. Index of Deprivation 2025


In [4]:
iod_raw = pd.read_csv(PATHS["iod2025"])

iod_source_columns = [
    "LSOA code (2021)",
    "LSOA name (2021)",
    "Local Authority District code (2024)",
    "Local Authority District name (2024)",
    "Index of Multiple Deprivation (IMD) Score",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)",
    "Income Score (rate)",
    "Income Rank (where 1 is most deprived)",
    "Income Decile (where 1 is most deprived 10% of LSOAs)",
]
require_columns(iod_raw, iod_source_columns, "IoD2025")

iod2025 = iod_raw[iod_source_columns].rename(columns={
    "LSOA code (2021)": "lsoa_code",
    "LSOA name (2021)": "lsoa_name",
    "Local Authority District code (2024)": "lad_code",
    "Local Authority District name (2024)": "lad_name",
    "Index of Multiple Deprivation (IMD) Score": "imd_score",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)": "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)": "imd_decile",
    "Income Score (rate)": "income_score",
    "Income Rank (where 1 is most deprived)": "income_rank",
    "Income Decile (where 1 is most deprived 10% of LSOAs)": "income_decile",
})

print("IoD2025 rows:", f"{len(iod2025):,}")
print("Duplicate LSOA codes:", iod2025["lsoa_code"].duplicated().sum())
display(iod2025.head())


IoD2025 rows: 33,755
Duplicate LSOA codes: 0


,lsoa_code,lsoa_name,lad_code,lad_name,imd_score,imd_rank,imd_decile,income_score,income_rank,income_decile
0,E01000001,City of London 001A,E09000001,City of London,8.742,26525,8,0.013,33730,10
1,E01000002,City of London 001B,E09000001,City of London,4.722,31203,10,0.018,33669,10
2,E01000003,City of London 001C,E09000001,City of London,9.250,25913,8,0.107,25167,8
3,E01000005,City of London 001E,E09000001,City of London,19.884,14807,5,0.211,14836,5
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,25.307,10917,4,0.343,7519,3


## 3. EVSE location registry

The source is connector-grain. The canonical hierarchy used later is:

`location_id` → `device_id` → `evse_uid` → `connector_id`.

No `drop_duplicates(..., keep="first")` reduction is performed here. The cleaning notebook
first checks that location-level attributes are consistent before constructing EVSE- and
location-level tables. Creation/deletion fields are retained for diagnostics, but they are not
used to remove records without confirmation of their exact source semantics.


In [5]:
evse_location_raw = pd.read_csv(PATHS["evse_location"], low_memory=False)

evse_required = [
    "Source", "id_x", "location_id_x", "location_id_y",
    "address", "city", "postal_code", "state", "location_class",
    "coordinates_latitude_y", "coordinates_longitude",
    "zapmap_device_uid", "power_band", "device_max_power",
    "evse_id", "evse_uid", "id_y", "operator_name",
    "created_at_x", "created_at_y", "created_at", "deleted_at_y",
]
require_columns(evse_location_raw, evse_required, "evse_location")

evse_location = evse_location_raw[evse_required].rename(columns={
    "Source": "source",
    "id_x": "location_id",
    "location_id_x": "location_id_x_source",
    "location_id_y": "location_id_y_source",
    "state": "borough",
    "location_class": "location_category",
    "coordinates_latitude_y": "latitude",
    "coordinates_longitude": "longitude",
    "zapmap_device_uid": "device_id",
    "id_y": "connector_id",
})

hierarchy_summary = pd.Series({
    "rows_connector_grain": len(evse_location),
    "unique_locations": evse_location["location_id"].nunique(),
    "unique_devices": evse_location["device_id"].nunique(),
    "unique_evses": evse_location["evse_uid"].nunique(),
    "unique_connectors": evse_location["connector_id"].nunique(),
})
print(hierarchy_summary.to_string())

print("\nCanonical location_id differs from location_id_x_source:",
      (~evse_location["location_id"].eq(evse_location["location_id_x_source"])).sum())
print("Canonical location_id differs from location_id_y_source:",
      (~evse_location["location_id"].eq(evse_location["location_id_y_source"])).sum())
print("evse_id differs from evse_uid:",
      (~evse_location["evse_id"].eq(evse_location["evse_uid"])).sum())
print("\nLocation categories:")
print(evse_location["location_category"].value_counts(dropna=False))


rows_connector_grain    51386
unique_locations        23013
unique_devices          32697
unique_evses            38358
unique_connectors       44941

Canonical location_id differs from location_id_x_source: 0
Canonical location_id differs from location_id_y_source: 14
evse_id differs from evse_uid: 0

Location categories:
location_category
On-street      36667
Destination    11442
En-route        1642
Other           1635
Name: count, dtype: int64


## 4. September 2025 charging activity


In [6]:
charging_activity_raw = pd.read_csv(PATHS["charging_activity"])
require_columns(
    charging_activity_raw,
    ["evse_uid", "start_time", "end_time", "duration"],
    "charging_activity_sep",
)

charging_activity = charging_activity_raw.rename(columns={
    "start_time": "charging_start",
    "end_time": "charging_end_reported",
    "duration": "charging_duration_min",
})

# Parsing here is for schema/date-range validation only. Cleaning decisions are made in 02.
charging_activity["charging_start"] = pd.to_datetime(
    charging_activity["charging_start"], errors="coerce"
)
charging_activity["charging_end_reported"] = pd.to_datetime(
    charging_activity["charging_end_reported"], errors="coerce"
)
charging_activity["charging_duration_min"] = pd.to_numeric(
    charging_activity["charging_duration_min"], errors="coerce"
)

print("Charging rows:", f"{len(charging_activity):,}")
print("Unique active EVSEs:", f"{charging_activity['evse_uid'].nunique():,}")
print("Start-time range:", charging_activity["charging_start"].min(),
      "to", charging_activity["charging_start"].max())
print("Missing parsed starts:", charging_activity["charging_start"].isna().sum())
print("Missing/invalid durations:", charging_activity["charging_duration_min"].isna().sum())
display(charging_activity.head())


Charging rows: 524,454
Unique active EVSEs: 18,748
Start-time range: 2025-09-01 00:00:00 to 2025-09-30 23:59:00
Missing parsed starts: 0
Missing/invalid durations: 0


,evse_uid,charging_start,charging_end_reported,charging_duration_min
0,0002729a6be20a87309637692e5a3af3,2025-09-07 22:21:00,2025-09-07 22:21:00,0.016667
1,0002729a6be20a87309637692e5a3af3,2025-09-07 22:21:00,2025-09-07 23:21:00,59.983333
2,0003081cf555fade0d74f080d8baa089,2025-09-11 21:49:00,2025-09-11 21:58:00,8.966667
3,0003081cf555fade0d74f080d8baa089,2025-09-23 09:08:00,2025-09-23 14:29:00,320.800000
4,00060c5bdc07754065222530a5987345,2025-09-04 17:11:00,2025-09-04 17:16:00,4.100000


## 5. Cross-file join coverage and loading summary


In [7]:
activity_uids = set(charging_activity["evse_uid"].dropna().astype(str))
registry_uids = set(evse_location["evse_uid"].dropna().astype(str))

print("Active EVSEs in charging file:", f"{len(activity_uids):,}")
print("EVSEs in registry:", f"{len(registry_uids):,}")
print("Active EVSEs absent from registry:", f"{len(activity_uids - registry_uids):,}")

loading_summary = pd.DataFrame({
    "dataset": ["TS045", "IoD2025", "evse_location", "charging_activity"],
    "rows": [len(ts045), len(iod2025), len(evse_location), len(charging_activity)],
    "columns": [ts045.shape[1], iod2025.shape[1], evse_location.shape[1], charging_activity.shape[1]],
    "raw_observation_unit": ["LSOA", "LSOA", "connector row", "charging session"],
})
display(loading_summary)


Active EVSEs in charging file: 18,748
EVSEs in registry: 38,358
Active EVSEs absent from registry: 0


,dataset,rows,columns,raw_observation_unit
0,TS045,35672,6,LSOA
1,IoD2025,33755,10,LSOA
2,evse_location,51386,22,connector row
3,charging_activity,524454,4,charging session
